# Validation 01 — File Scanner & Sensor Ordering
Verifies WAV files are found and sorted accelerometers-first.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, warnings
from pathlib import Path
from scipy.signal import butter,cheby1,cheby2,ellip,bessel,sosfiltfilt,sosfreqz,welch
from scipy.signal import spectrogram as sp_spectrogram
from scipy.io import wavfile
from itertools import product
warnings.filterwarnings('ignore')

BASE_DIR    = Path(r"D:\\1 placement\\IAESTE INTERNSHIP CZECH\\iaeste26-blasting-sound-main\\iaeste26-blasting-sound-main")
DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_PRIORITY   = ['AccAxial4507','AccRadial4507','Mic147EB','Mic46BE']
ENERGY_WINDOW_S   = 0.05;  NOISE_DURATION_S = 0.5;  ONSET_THRESHOLD = 10.0;  ONSET_OFFSET_S = 7.0
WINDOW_DURATION_S = 5.0;   WINDOW_STEP_S    = 1.0
LOWER_LIMITS = [176,225,283,353,440,565,707,880,1130,1414,1760,10,10,500,1000]
UPPER_LIMITS = [225,283,353,440,565,707,880,1130,1414,1760,2220,1000,2000,1500,2000]
N_BANDS      = len(LOWER_LIMITS)
BAND_LABELS  = [f"{lo}–{hi} Hz" for lo,hi in zip(LOWER_LIMITS,UPPER_LIMITS)]
FILTER_TYPES  = ['Butterworth','Chebyshev I','Chebyshev II','Elliptical','Bessel']
FILTER_ORDERS = [3,5,7]
CHEBY1_RIPPLE_DB=0.5; CHEBY2_ATTEN_DB=40.0; ELLIP_RIPPLE_DB=0.5; ELLIP_ATTEN_DB=40.0

P=[0]; F=[0]
def check(label, ok, note=""):
    s="[PASS]" if ok else "[FAIL]"
    if ok: P[0]+=1
    else:  F[0]+=1
    print(f"  {s}  {label}" + (f"  → {note}" if note else ""))
def info(label, val): print(f"  [INFO]  {label}: {val}")
def summary():
    t=P[0]+F[0]
    print(f"\n{'='*50}")
    print(f"  PASS: {P[0]}/{t}  |  FAIL: {F[0]}/{t}")
    print(f"  Score: {P[0]/t*100:.0f}%" if t else "  No checks run")
    print('='*50)
print("Config loaded.")


Config loaded.


In [2]:
all_wavs = sorted(set(DATA_DIR.rglob("*.wav"))|set(DATA_DIR.rglob("*.WAV")))
def skey(p):
    s=Path(p).stem.split('_')[-1]
    return SENSOR_PRIORITY.index(s) if s in SENSOR_PRIORITY else 99
metas = sorted([{'path':p,'name':p.name,'sensor':p.stem.split('_')[-1]} for p in all_wavs], key=lambda m: skey(m['path']))
sensors = [m['sensor'] for m in metas]
accel_i = [i for i,s in enumerate(sensors) if 'Acc' in s]
mic_i   = [i for i,s in enumerate(sensors) if 'Mic' in s]

check("DATA_DIR exists",               DATA_DIR.exists(),      str(DATA_DIR))
check("At least 1 WAV file found",     len(all_wavs)>0,        f"{len(all_wavs)} files")
check("4 sensors in SENSOR_PRIORITY",  len(SENSOR_PRIORITY)==4)
check("AccAxial4507 first in priority",SENSOR_PRIORITY[0]=='AccAxial4507')
check("Mic46BE last in priority",      SENSOR_PRIORITY[-1]=='Mic46BE')
if accel_i and mic_i:
    check("All accelerometers before all microphones",
          max(accel_i)<min(mic_i), f"last accel pos {max(accel_i)}, first mic pos {min(mic_i)}")
else:
    info("Only one sensor type found", sensors[:3])
check("RESULTS_DIR created",  (RESULTS_DIR.mkdir(parents=True,exist_ok=True) or True) and RESULTS_DIR.exists())

print(f"\nFile list (sorted):")
for m in metas: print(f"  {m['sensor']:<18} {m['name']}")
summary()


  [PASS]  DATA_DIR exists  → D:\1 placement\IAESTE INTERNSHIP CZECH\iaeste26-blasting-sound-main\iaeste26-blasting-sound-main\data
  [PASS]  At least 1 WAV file found  → 1568 files
  [PASS]  4 sensors in SENSOR_PRIORITY
  [PASS]  AccAxial4507 first in priority
  [PASS]  Mic46BE last in priority
  [PASS]  All accelerometers before all microphones  → last accel pos 783, first mic pos 784
  [PASS]  RESULTS_DIR created

File list (sorted):
  AccAxial4507       G80_8_3_0_AccAxial4507.wav
  AccAxial4507       G80_8_3_100_AccAxial4507.wav
  AccAxial4507       G80_8_3_30_AccAxial4507.wav
  AccAxial4507       G80_8_3_40_AccAxial4507.wav
  AccAxial4507       G80_8_3_50_AccAxial4507.wav
  AccAxial4507       G80_8_3_60_AccAxial4507.wav
  AccAxial4507       G80_8_3_70_AccAxial4507.wav
  AccAxial4507       G80_8_4_0_AccAxial4507.wav
  AccAxial4507       G80_8_4_100_AccAxial4507.wav
  AccAxial4507       G80_8_4_30_AccAxial4507.wav
  AccAxial4507       G80_8_4_40_AccAxial4507.wav
  AccAxial4507       